# TurkishBERTweet - Siber Zorbalık Sınıflandırması
Bu notebook, **TurkishBERTweet** modeli ile siber zorbalık tespiti yapmayı hedefler.

Adil bir kıyaslama için DOCX makalesindeki kriterlere **birebir uygun** olarak kodlanmıştır:
- Epoch: 4
- Learning Rate: 2e-5
- Batch Size: 16
- Max Sequence Length: 256
- Eğitim / Test Ayrımı: %80 / %20

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import torch
import gc
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'*** Kullanılan Cihaz: {device} ***')

*** Kullanılan Cihaz: cuda ***


In [ ]:
# -----------------------------------------------------
# 1. MODEL VE HİPERPARAMETRELER 
# -----------------------------------------------------
EPOCHS = 4
LEARNING_RATE = 2e-5
BATCH_SIZE = 16
MAX_LEN = 256
MODEL_NAME = 'VRLLab/TurkishBERTweet'
RANDOM_STATE = 42

In [ ]:
# -----------------------------------------------------
# 2. METİN ÖN İŞLEME 
# -----------------------------------------------------
import re

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)   # URL kaldır
    text = re.sub(r'@\w+', '', text)                        # @mention kaldır
    text = re.sub(r'#\w+', '', text)                        # #hashtag kaldır
    text = re.sub(r'[^\w\s]', '', text)                     # Noktalama kaldır
    text = re.sub(r'\s+', ' ', text).strip()                # Fazla boşluk temizle
    return text

# -----------------------------------------------------
# 3. DATASET SINIFI
# -----------------------------------------------------
class CyberbullyingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        encoding = self.tokenizer(
            text, add_special_tokens=True, max_length=self.max_len,
            return_token_type_ids=False, padding='max_length',
            truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'targets': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# -----------------------------------------------------
# 4. EĞİTİM VE TEST FONKSİYONLARI 
# -----------------------------------------------------
import copy, random
from transformers import get_linear_schedule_with_warmup

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def train_epoch(model, data_loader, optimizer, device, scheduler):
    model = model.train()
    losses = []
    correct_predictions = 0
    total_examples = 0
    
    for d in data_loader:
        input_ids = d['input_ids'].to(device)
        attention_mask = d['attention_mask'].to(device)
        targets = d['targets'].to(device)

        # Hugging Face'in kendi Loss hesaplamasını kullanıyoruz (labels parametresi ile)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=targets)
        loss = outputs.loss
        logits = outputs.logits
        
        _, preds = torch.max(logits, dim=1)
        correct_predictions += torch.sum(preds == targets)
        total_examples += len(targets)
        losses.append(loss.item())
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
    return correct_predictions.double() / total_examples, np.mean(losses)

def eval_model(model, data_loader, device):
    model = model.eval()
    losses = []
    correct_predictions = 0
    total_examples = 0
    real_targets, predictions = [], []

    with torch.no_grad():
        for d in data_loader:
            input_ids = d['input_ids'].to(device)
            attention_mask = d['attention_mask'].to(device)
            targets = d['targets'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=targets)
            loss = outputs.loss
            logits = outputs.logits
            
            _, preds = torch.max(logits, dim=1)

            correct_predictions += torch.sum(preds == targets)
            total_examples += len(targets)
            losses.append(loss.item())
            
            real_targets.extend(targets.cpu().numpy())
            predictions.extend(preds.cpu().numpy())

    accuracy = accuracy_score(real_targets, predictions)
    # average='macro' YERİNE average='weighted' KULLANILDI
    precision, recall, f1, _ = precision_recall_fscore_support(real_targets, predictions, average='weighted', zero_division=0)
    return accuracy, precision, recall, f1, np.mean(losses)

def run_experiment(file_path):
    set_seed(RANDOM_STATE)
    print(f'\n========== [ MESAİ BAŞLADI: {file_path} ] ==========')
    df = pd.read_excel(file_path)

    text_col = df.columns[0]
    label_col = 'total' if 'total' in df.columns else df.columns[1]
    df = df.dropna(subset=[text_col, label_col])

    # Makale ön işlemesi uygulandı
    df[text_col] = df[text_col].apply(preprocess_text)

    df_train, df_test = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE, stratify=df[label_col])
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    train_data_loader = DataLoader(
        CyberbullyingDataset(df_train[text_col].to_numpy(), df_train[label_col].to_numpy(), tokenizer, MAX_LEN),
        batch_size=BATCH_SIZE, shuffle=True
    )
    test_data_loader = DataLoader(
        CyberbullyingDataset(df_test[text_col].to_numpy(), df_test[label_col].to_numpy(), tokenizer, MAX_LEN),
        batch_size=BATCH_SIZE, shuffle=False
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, classifier_dropout=0.1 # Dropout makalede geçmiyorsa standart bırakıldı
    ).to(device)

    # Standart AdamW kullanıldı 
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, eps=1e-8)

    total_steps = len(train_data_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * 0.1),
        num_training_steps=total_steps
    )

    best_f1 = 0
    best_model_weights = None

    for epoch in range(EPOCHS):
        print(f'Epoch {epoch + 1}/{EPOCHS} çalışıyor...')
        train_acc, train_loss = train_epoch(model, train_data_loader, optimizer, device, scheduler)
        val_acc, val_precision, val_recall, val_f1, val_loss = eval_model(model, test_data_loader, device)
        print(f'  ✓ Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Test F1: {val_f1:.4f}')

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_model_weights = copy.deepcopy(model.state_dict())

    if best_model_weights is not None:
        model.load_state_dict(best_model_weights)
        print("💡 En iyi Epoch ağırlıkları geri yüklendi!")

    accuracy, precision, recall, f1, test_loss = eval_model(model, test_data_loader, device)

    print(f'\n🔥 FİNAL SONUÇ METRİKLERİ ({file_path}):')
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print("="*60)

    del model, optimizer, scheduler, train_data_loader, test_data_loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return accuracy, precision, recall, f1

In [ ]:
pre_covid_metrics = run_experiment('/content/drive/MyDrive/data/covidoncesi.xlsx')
post_covid_metrics = run_experiment('/content/drive/MyDrive/data/covidsonrasi.xlsx')
combined_metrics = run_experiment('/content/drive/MyDrive/data/covidoncesivesonrasi.xlsx')


========== [ MESAİ BAŞLADI: /content/drive/MyDrive/data/covidoncesi.xlsx ] ==========


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/352 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

RobertaForSequenceClassification LOAD REPORT from: VRLLab/TurkishBERTweet
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/4 çalışıyor...
  ✓ Train Loss: 0.6278 | Train Acc: 0.6537 | Test F1: 0.6631
Epoch 2/4 çalışıyor...
  ✓ Train Loss: 0.5160 | Train Acc: 0.7647 | Test F1: 0.6719
Epoch 3/4 çalışıyor...
  ✓ Train Loss: 0.2979 | Train Acc: 0.8878 | Test F1: 0.6395
Epoch 4/4 çalışıyor...
  ✓ Train Loss: 0.1778 | Train Acc: 0.9462 | Test F1: 0.6332
💡 En iyi Epoch ağırlıkları geri yüklendi!

🔥 FİNAL SONUÇ METRİKLERİ (/content/drive/MyDrive/data/covidsonrasi.xlsx):
Accuracy:  0.7024
Precision: 0.6903
Recall:    0.7024
F1-Score:  0.6719

========== [ MESAİ BAŞLADI: /content/drive/MyDrive/data/covidoncesivesonrasi.xlsx ] ==========


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: VRLLab/TurkishBERTweet
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/4 çalışıyor...
  ✓ Train Loss: 0.6097 | Train Acc: 0.6761 | Test F1: 0.7200
Epoch 2/4 çalışıyor...
  ✓ Train Loss: 0.4684 | Train Acc: 0.7942 | Test F1: 0.7065
Epoch 3/4 çalışıyor...
  ✓ Train Loss: 0.2593 | Train Acc: 0.9031 | Test F1: 0.6628
Epoch 4/4 çalışıyor...
  ✓ Train Loss: 0.1559 | Train Acc: 0.9560 | Test F1: 0.6630
💡 En iyi Epoch ağırlıkları geri yüklendi!

🔥 FİNAL SONUÇ METRİKLERİ (/content/drive/MyDrive/data/covidoncesivesonrasi.xlsx):
Accuracy:  0.7296
Precision: 0.7194
Recall:    0.7296
F1-Score:  0.7200
